# Project 1: Basic input guardrails

## 1. What can happen?

Someone sends a blank message or a very long one. What should our app do before it tries to respond?

## 2. Our idea

Trim spaces from both ends. Block the message if it is blank or over 40 characters.

## 3. How it helps

A blocked message gets a reason, and the responder never runs. An allowed message goes to a small Python function that returns a demo reply. No API key is needed.

We'll try the same messages again with an 80-character limit and see what changes.

## 4. Input and setup

The [example inputs](docs/dataset/inputs.json) include a short leave question, an empty message, a message made of spaces and other whitespace, a 100-character string, and a longer leave question.

Run `uv sync --locked` from the repository root. In VS Code, open `1_basic_input_guardrails` as a folder and select the repository's `.venv/Scripts/python.exe` kernel.


### Step 1. Check the setup

Run the next cell to check the folder and Python kernel. It also loads the input check, the function that calls the responder, and the helpers that run and display the examples.


In [ ]:
import json
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd()
assert PROJECT_DIR.name == "1_basic_input_guardrails", "Use this numbered folder as the working directory."
assert Path(sys.prefix).resolve() == (PROJECT_DIR.parent / ".venv").resolve(), "Select the shared root .venv kernel."
from tools.input_guard import check_input
from tools.guarded_request import run_request
from tools.examples import run_examples, show_results, show_comparison
print("Project directory and shared kernel are ready. No API keys are needed.")


### Step 2. Look at the example messages

Compare each message's length before and after trimming. The whitespace-only message becomes empty.


In [ ]:
dataset = json.loads((PROJECT_DIR / "docs/dataset/inputs.json").read_text(encoding="utf-8"))
examples = dataset["cases"]
print(dataset["provenance"])
for example in examples:
    print(example["case_id"], repr(example["input"]), "raw:", len(example["input"]), "trimmed:", len(example["input"].strip()))


### Step 3. Try a 40-character limit

The short leave question should get a reply. Blank and long messages should be blocked. Look for the reason and how many times the demo responder ran for each message.


In [ ]:
baseline_limit = 40
baseline = run_examples(examples, max_chars=baseline_limit)
show_results(baseline)
print("Allowed request response:", baseline[0]["response"])


### Step 4. Try 40 and 41 characters

The 40-character message should pass. One extra character should make it block.


In [ ]:
for text in ("A" * 40, "A" * 41):
    result = check_input(text, max_chars=baseline_limit)
    print(len(text), result["decision"], result["reason"])


### Step 5. Look at the longer leave question

This is a reasonable question, but it goes over 40 characters. Read the reason the guard gives.


In [ ]:
longer = next(example for example in examples if example["case_id"] == "longer_legitimate")
print(longer["input"])
print("Trimmed characters:", len(longer["input"].strip()))
print(run_request(longer["input"], max_chars=baseline_limit))


### Step 6. Try an 80-character limit

Run the same messages again. The longer leave question should now pass. The blank messages and the 100-character string should still be blocked.


In [ ]:
candidate_limit = 80
candidate = run_examples(examples, max_chars=candidate_limit)
show_results(candidate)
show_comparison(baseline, candidate)
print("Longer request response:", next(row for row in candidate if row["case_id"] == "longer_legitimate")["response"])


### Step 7. Check when the responder ran

Each blocked message should show zero calls and no reply. Each allowed message should show one call to the local demo responder.


In [ ]:
for row in baseline + candidate:
    expected_local_calls = 1 if row["allowed"] else 0
    assert row["local_responder_calls"] == expected_local_calls
    assert (row["response"] is not None) == row["allowed"]
    assert row["model_api_calls"] == 0
print("Displayed requests agree with the guard decisions and local call counts.")


### Stop and explain

Which message changed from blocked to allowed? Would a short harmful message pass this length check?
